In [2]:
!nvidia-smi

Wed Nov 18 17:11:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
print("Token loaded ✅")

Token loaded ✅


In [4]:
!pip install -q --upgrade kaggle

In [5]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   85M  113G   1% /


In [6]:
!kaggle datasets download -d awsaf49/brats20-dataset-training-validation -p /content
!ls -lh /content

Dataset URL: https://www.kaggle.com/datasets/awsaf49/brats20-dataset-training-validation
License(s): CC0-1.0
100% 4.16G/4.16G [00:36<00:00, 122MB/s]

total 4.2G
-rw-r--r-- 1 root root 4.2G Jul  2  2020 brats20-dataset-training-validation.zip
drwxr-xr-x 1 root root 4.0K Nov 16 15:56 sample_data


In [7]:
!unzip -q /content/brats20-dataset-training-validation.zip "BraTS2020_TrainingData/*" -d /content/brats

In [8]:
import os

root = "/content/brats/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
patients = sorted([p for p in os.listdir(root) if p.startswith("BraTS20")])
print("Patients:", len(patients))

for p in patients:
    files = os.listdir(f"{root}/{p}")
    if f"{p}_seg.nii" not in files:
        print("⚠️ Seg missing/peyar vera:", p, files)

Patients: 369
⚠️ Seg missing/peyar vera: BraTS20_Training_355 ['BraTS20_Training_355_t1ce.nii', 'BraTS20_Training_355_flair.nii', 'W39_1998.09.19_Segm.nii', 'BraTS20_Training_355_t2.nii', 'BraTS20_Training_355_t1.nii']


In [9]:
old = f"{root}/BraTS20_Training_355/W39_1998.09.19_Segm.nii"
new = f"{root}/BraTS20_Training_355/BraTS20_Training_355_seg.nii"

if os.path.exists(old):
    os.rename(old, new)
    print("Renamed ✅")

Renamed ✅


In [10]:
import os

root = "/content/brats/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
patients = sorted([p for p in os.listdir(root) if p.startswith("BraTS20")])
print("Patients:", len(patients))

for p in patients:
    files = os.listdir(f"{root}/{p}")
    if f"{p}_seg.nii" not in files:
        print("⚠️ Seg missing/peyar vera:", p, files)

Patients: 369


In [11]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

MODALITIES = ["flair", "t1", "t1ce", "t2"]

def normalize(vol):
    brain = vol > 0                          # brain voxels mattum (True/False map)
    mean = vol[brain].mean()
    std = vol[brain].std()
    out = np.zeros_like(vol, dtype=np.float32)
    out[brain] = (vol[brain] - mean) / std   # z = (x − μ) ÷ σ
    return out

def load_patient(p):
    folder = f"{root}/{p}"
    scans = []
    for m in MODALITIES:
        vol = nib.load(f"{folder}/{p}_{m}.nii").get_fdata()
        scans.append(normalize(vol))
    image = np.stack(scans, axis=0)          # 4 scans → (4, 240, 240, 155)

    seg = nib.load(f"{folder}/{p}_seg.nii").get_fdata().astype(np.uint8)
    seg[seg == 4] = 3                        # labels: 0,1,2,4 → 0,1,2,3

    image = image[:, 56:184, 56:184, 13:141] # crop
    seg = seg[56:184, 56:184, 13:141]
    return image, seg

image, seg = load_patient("BraTS20_Training_001")

print("Image shape:", image.shape)
print("Seg shape  :", seg.shape)
print("Seg labels :", np.unique(seg))
flair = image[0]
print(f"FLAIR brain mean: {flair[flair != 0].mean():.3f}, std: {flair[flair != 0].std():.3f}")
print("Tumour voxels after crop:", np.sum(seg > 0))

Image shape: (4, 128, 128, 128)
Seg shape  : (128, 128, 128)
Seg labels : [0 1 2 3]
FLAIR brain mean: 0.044, std: 1.035
Tumour voxels after crop: 211033


In [12]:
from tqdm import tqdm

out_dir = "/content/preprocessed"
os.makedirs(out_dir, exist_ok=True)          # folder create (irundhaa paravaayilla)

crop_loss = []

for p in tqdm(patients):
    image, seg = load_patient(p)
    np.save(f"{out_dir}/{p}_image.npy", image.astype(np.float16))
    np.save(f"{out_dir}/{p}_seg.npy", seg)

    # crop evvalavu tumour-ai vettudhu?
    full_seg = nib.load(f"{root}/{p}/{p}_seg.nii").get_fdata()
    full = np.sum(full_seg > 0)
    kept = np.sum(seg > 0)
    crop_loss.append((p, 100 * (full - kept) / full))

losses = np.array([l for _, l in crop_loss])
print(f"\nAverage crop loss: {losses.mean():.3f} %")
print(f"Max crop loss    : {losses.max():.3f} %")
print(f"Patients with > 1% loss: {np.sum(losses > 1)}")

print("\nWorst 5:")
for p, l in sorted(crop_loss, key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {p}: {l:.2f} %")

100%|██████████| 369/369 [10:50<00:00,  1.76s/it]


Average crop loss: 4.104 %
Max crop loss    : 61.411 %
Patients with > 1% loss: 148

Worst 5:
  BraTS20_Training_314: 61.41 %
  BraTS20_Training_277: 48.31 %
  BraTS20_Training_366: 44.35 %
  BraTS20_Training_181: 40.87 %
  BraTS20_Training_217: 38.75 %


In [13]:
!du -sh /content/preprocessed
!ls /content/preprocessed | wc -l

6.5G	/content/preprocessed
738


In [14]:
lo = [56, 56, 13]      # crop start (x, y, z)
hi = [184, 184, 141]   # crop end

mins, maxs = [], []
for p in tqdm(patients):
    s = nib.load(f"{root}/{p}/{p}_seg.nii").get_fdata()
    idx = np.argwhere(s > 0)              # ellaa tumour voxels-oda (x, y, z) positions
    mins.append(idx.min(axis=0))          # tumour-oda chinna x, y, z
    maxs.append(idx.max(axis=0))          # tumour-oda periya x, y, z

mins, maxs = np.array(mins), np.array(maxs)

for a, name in enumerate(["x", "y", "z"]):
    outside = np.sum((mins[:, a] < lo[a]) | (maxs[:, a] >= hi[a]))
    print(f"{name}: tumour range {mins[:, a].min()} – {maxs[:, a].max()} | "
          f"crop {lo[a]}–{hi[a]} | patients outside: {outside}")

100%|██████████| 369/369 [00:29<00:00, 12.40it/s]

x: tumour range 45 – 239 | crop 56–184 | patients outside: 96
y: tumour range 29 – 215 | crop 56–184 | patients outside: 192
z: tumour range 15 – 143 | crop 13–141 | patients outside: 6


In [15]:
lo = [40, 24, 8]
hi = [200, 216, 152]

for a, name in enumerate(["x", "y", "z"]):
    outside = np.sum((mins[:, a] < lo[a]) | (maxs[:, a] >= hi[a]))
    print(f"{name}: crop {lo[a]}–{hi[a]} (size {hi[a]-lo[a]}) | patients outside: {outside}")

# x-la andha 239 yaaru?
print("\nx max top 5:", sorted(maxs[:, 0])[-5:])
print("Patients with x max > 200:", [patients[i] for i in np.where(maxs[:, 0] >= 200)[0]])

x: crop 40–200 (size 160) | patients outside: 2
y: crop 24–216 (size 192) | patients outside: 0
z: crop 8–152 (size 144) | patients outside: 0

x max top 5: [np.int64(191), np.int64(191), np.int64(193), np.int64(212), np.int64(239)]
Patients with x max > 200: ['BraTS20_Training_091', 'BraTS20_Training_344']


In [16]:
for p in ["BraTS20_Training_091", "BraTS20_Training_344"]:
    s = nib.load(f"{root}/{p}/{p}_seg.nii").get_fdata()
    out = np.sum(s[200:, :, :] > 0)      # x = 200 mudhal 239 varaikkum
    total = np.sum(s > 0)
    print(f"{p}: {out} voxels outside of {total} ({100 * out / total:.3f} %)")

BraTS20_Training_091: 4 voxels outside of 149594 (0.003 %)
BraTS20_Training_344: 18 voxels outside of 79098 (0.023 %)


In [17]:
MODALITIES = ["flair", "t1", "t1ce", "t2"]
X0, X1 = 40, 200     # 160
Y0, Y1 = 24, 216     # 192
Z0, Z1 = 8, 152      # 144

def normalize(vol):
    brain = vol > 0
    out = np.zeros_like(vol, dtype=np.float32)
    out[brain] = (vol[brain] - vol[brain].mean()) / vol[brain].std()
    return out

def load_patient(p):
    folder = f"{root}/{p}"
    scans = [normalize(nib.load(f"{folder}/{p}_{m}.nii").get_fdata()) for m in MODALITIES]
    image = np.stack(scans, axis=0)

    seg = nib.load(f"{folder}/{p}_seg.nii").get_fdata().astype(np.uint8)
    seg[seg == 4] = 3
    full = np.sum(seg > 0)                       # crop-ku munnaadi tumour count

    image = image[:, X0:X1, Y0:Y1, Z0:Z1]
    seg = seg[X0:X1, Y0:Y1, Z0:Z1]
    return image, seg, full

out_dir = "/content/preprocessed"
os.makedirs(out_dir, exist_ok=True)

loss = []
for p in tqdm(patients):
    image, seg, full = load_patient(p)
    np.save(f"{out_dir}/{p}_image.npy", image.astype(np.float16))
    np.save(f"{out_dir}/{p}_seg.npy", seg)
    loss.append(100 * (full - np.sum(seg > 0)) / full)

loss = np.array(loss)
print("Shape:", image.shape, seg.shape)
print(f"Avg loss: {loss.mean():.4f} % | Max: {loss.max():.4f} % | >1%: {np.sum(loss > 1)}")
!du -sh {out_dir}

100%|██████████| 369/369 [13:28<00:00,  2.19s/it]

Shape: (4, 160, 192, 144) (160, 192, 144)
Avg loss: 0.0001 % | Max: 0.0228 % | >1%: 0
14G	/content/preprocessed
